# **E-Commerce Analytics & Data Engineering System**

### **1. Spark Application Setup**
The application must initialize a valid environment:

* Create a SparkSession with a clear application name.
* Explicitly configure the master URL to local mode.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
                    .appName('spark_assignment')\
                    .master('local[*]')\
                    .getOrCreate()

### **2. Data Ingestion & Validation**
* **Validation Logic**: Separate valid records from invalid/malformed records.
* **Audit**: Report the total number of records vs. incomplete records.
* **Schema**: Enforce an explicit schema (StructType) during the ingestion of valid data.

In [2]:
# for products.csv
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DoubleType, TimestampType

# schema for products.csv
product_schema = StructType([
    StructField('product_id', StringType(), True),
    StructField('product_name', StringType(), True),
    StructField('brand', StringType(), True)
])

# creating dataframe for total number of products
total_products_df = spark.read.format('csv')\
                              .option('header', True)\
                              .schema(product_schema)\
                              .option('mode', 'permissive')\
                              .load('products.csv')

# creating dataframe for valid products only
valid_products_df = spark.read.format('csv')\
                              .option('header', True)\
                              .schema(product_schema)\
                              .option('mode', 'dropmalformed')\
                              .load('products.csv')

print('Total Number of Products :- ' + str(total_products_df.count()))
print('Number of Products that are valid :- ' + str(valid_products_df.count()))
print('Number of Products that are invalid/malformed :- ' + str(total_products_df.count() - valid_products_df.count()))

Total Number of Products :- 100
Number of Products that are valid :- 100
Number of Products that are invalid/malformed :- 0


In [3]:
# for transactions.csv

# schema for transactions.csv
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("transaction_timestamp", TimestampType(), True),
    StructField("platform", StringType(), True),
    StructField("country", StringType(), True)
])

# creating dataframe for total number of transactions
total_transactions_df = spark.read.format('csv')\
                              .option('header', True)\
                              .schema(transaction_schema)\
                              .option('mode', 'permissive')\
                              .load('transactions.csv')

# creating dataframe for valid transactions only
valid_transactions_df = spark.read.format('csv')\
                              .option('header', True)\
                              .schema(transaction_schema)\
                              .option('mode', 'dropmalformed')\
                              .load('transactions.csv')

print('Total Number of transactions :- ' + str(total_transactions_df.count()))
print('Number of transactions that are valid :- ' + str(valid_transactions_df.count()))
print('Number of transactions that are invalid/malformed :- ' + str(total_transactions_df.count() - valid_transactions_df.count()))

Total Number of transactions :- 214
Number of transactions that are valid :- 214
Number of transactions that are invalid/malformed :- 0


In [4]:
# making use of accumulators for counting invalid records
invalid_counts = spark.sparkContext.accumulator(0)

def invalid_record(row):
    if row.price is None:
        invalid_counts.add(1)
        return False
    return True

df_filtered = total_transactions_df.rdd.filter(invalid_record)
df_filtered.count()

print("Invalid records:", invalid_counts.value)

Invalid records: 5


### **3. Data Cleaning & Standardization**
Apply transformations to ensure data readiness:

* **Null Handling:** Remove or impute missing values.
* **Deduplication:** Identify and remove duplicate transaction records.
* **Standardization:** Normalize inconsistent category values and cast columns to correct numeric/timestamp types.

In [5]:
# now we will use only valid_products_df and valid_transactions_df

'''
various options for dropping
* df.dropna(how="all")
* df.dropna(how="any")
* df.dropna(subset=["price"])
* df.dropna(thresh=2)
'''

# null handling
valid_pro_df_null_removed = valid_products_df.dropna()
print('total entries before dropping null values in products.csv:- ' + str(valid_products_df.count()))
print('total entries after dropping null values in products.csv:- ' + str(valid_pro_df_null_removed.count()))
valid_tra_df_null_removed = valid_transactions_df.dropna()
print('total entries before dropping null values in transactions.csv:- ' + str(valid_transactions_df.count()))
print('total entries after dropping null values in transactions.csv:- ' + str(valid_tra_df_null_removed.count()))

print(' ')

# deduplication
valid_pro_df_deduplicated = valid_pro_df_null_removed.drop_duplicates()
print('total entries before De-Duplicating in products.csv:- ' + str(valid_pro_df_null_removed.count()))
print('total entries after De-Duplicating in products.csv:- ' + str(valid_pro_df_deduplicated.count()))
valid_tra_df_deduplicated = valid_tra_df_null_removed.drop_duplicates()
print('total entries before De-Duplicating in transactions.csv:- ' + str(valid_tra_df_null_removed.count()))
print('total entries after De-Duplicating in transactions.csv:- ' + str(valid_tra_df_deduplicated.count()))

total entries before dropping null values in products.csv:- 100
total entries after dropping null values in products.csv:- 100
total entries before dropping null values in transactions.csv:- 214
total entries after dropping null values in transactions.csv:- 198
 
total entries before De-Duplicating in products.csv:- 100
total entries after De-Duplicating in products.csv:- 100
total entries before De-Duplicating in transactions.csv:- 198
total entries after De-Duplicating in transactions.csv:- 195


In [6]:
# standardization
from pyspark.sql.functions import col, initcap, trim, lower, to_timestamp
df_transactions_cleaned = valid_tra_df_deduplicated.withColumn('category', initcap(trim(lower(col('category')))))\
                                                   .withColumn('price', col('price').cast('double'))\
                                                   .withColumn('quantity', col('quantity').cast('int'))\
                                                   .withColumn('transaction_timestamp', to_timestamp(col('transaction_timestamp'), 'yyyy-MM-dd HH:mm:ss'))

df_products_cleaned = valid_pro_df_deduplicated

df_products_cleaned.show()

print(' ')

df_transactions_cleaned.show()

+----------+--------------------+-------------+
|product_id|        product_name|        brand|
+----------+--------------------+-------------+
|   PROD034|       Golf Club Set|      ProGolf|
|   PROD018|        Evening Gown|  ElegantWear|
|   PROD053|History Book Coll...|TimelessReads|
|   PROD043|        Sweater Wool|     CozyWear|
|   PROD098| Blazer Professional|   CareerWear|
|   PROD007|Casual Cotton T-S...|    StyleWear|
|   PROD040|Dumbbells Adjustable|      FitGear|
|   PROD089|   Webcam Ring Light|    StreamPro|
|   PROD025|Biography Collection|     ReadMore|
|   PROD074|         Garden Hose|   WaterWorks|
|   PROD081|      Baseball Glove|  DiamondGear|
|   PROD016| Running Shoes Elite|    ActiveFit|
|   PROD028|  Basketball Premium|   SportsGear|
|   PROD042|    Portable SSD 2TB|    DataStore|
|   PROD026|Wireless Earbuds Pro|   AudioElite|
|   PROD062|          VR Headset| VirtualWorld|
|   PROD051|      Skateboard Pro|ExtremeSports|
|   PROD033|    Formal Suit Navy| Busine

### **4. Partitioning & Performance**
Demonstrate awareness of Spark’s distributed architecture:

* Inspect and modify the number of partitions to optimize parallel processing.
* Use coalesce() or repartition() appropriately to reduce partitions before writing.
* Implement caching or persistence only for DataFrames reused in multiple actions.

In [7]:
# identifying and modifying number of partitions to optimize parellel processing
print('total number of products dataframe partitions before:- ', df_products_cleaned.rdd.getNumPartitions())
print('total number of transactions dataframe partitions before:- ', df_transactions_cleaned.rdd.getNumPartitions())

df_products_cleaned = df_products_cleaned.repartition(8)
df_transactions_cleaned = df_transactions_cleaned.repartition(8)

print('total number of products dataframe partitions set to :- ', df_products_cleaned.rdd.getNumPartitions())
print('total number of transactions dataframe partitions set to:- ', df_transactions_cleaned.rdd.getNumPartitions())


total number of products dataframe partitions before:-  1
total number of transactions dataframe partitions before:-  1
total number of products dataframe partitions set to :-  8
total number of transactions dataframe partitions set to:-  8


### **5. Business Analytics & Aggregations**
Generate the following reports using distributed operations:

* Total Revenue: Calculated per country.
* Top Performers: Identify the top 5 products by total revenue.
* Platform Metrics: Average order value per platform.
* Trends: Daily transaction count trends.

In [8]:
# Total Revenue: Calculated per country.
import pyspark.sql.functions as F

print('Total Revenue: Calculated per country. \n')
df_transactions_cleaned.groupBy("country") \
    .agg(
        F.sum(col("price") * col("quantity")).alias("total_revenue")
    ) \
    .show()

Total Revenue: Calculated per country. 

+-------+------------------+
|country|     total_revenue|
+-------+------------------+
|Germany|10898.769999999999|
| France|23598.709999999995|
|    USA|          33045.79|
|     UK|          16451.73|
| Canada|24937.340000000004|
+-------+------------------+



In [9]:
# Top Performers: Identify the top 5 products by total revenue.
import pyspark.sql.functions as F

print('Identify the top 5 products by total revenue. \n')

top_products = df_transactions_cleaned.groupBy("product_id") \
    .agg(
        F.sum(col("price") * col("quantity")).alias("total_revenue")
    ) \
    .orderBy(col('total_revenue').desc())\
    .limit(5)

top_products_with_name = top_products.join(df_products_cleaned, on = 'product_id', how = 'inner')

top_products_with_name.select(
    'product_id',
    'product_name',
    'total_revenue'
).show(truncate=False)

Identify the top 5 products by total revenue. 

+----------+-----------------------+-------------+
|product_id|product_name           |total_revenue|
+----------+-----------------------+-------------+
|PROD092   |Projector Home Cinema  |3399.98      |
|PROD086   |All-in-One Desktop     |3199.98      |
|PROD046   |Gaming Monitor 27 inch |3599.98      |
|PROD032   |Gaming Console Next Gen|3199.98      |
|PROD076   |Laptop Gaming Elite    |3999.98      |
+----------+-----------------------+-------------+



In [10]:
# Platform Metrics: Average order value per platform.
from pyspark.sql.functions import avg

print('Average order value per platform. \n')

df_transactions_cleaned.groupBy('platform')\
    .agg(avg('price').alias('avg_value')).show()

Average order value per platform. 

+--------+------------------+
|platform|         avg_value|
+--------+------------------+
|  Mobile|309.26635416666664|
|     Web| 600.3084848484848|
+--------+------------------+



In [11]:
# Trends: Daily transaction count trends.
from pyspark.sql.functions import to_date, count

df_transactions_cleaned.withColumn('date', to_date(col('transaction_timestamp')))\
                       .groupBy('date')\
                       .agg(count('*'). alias('transaction_count'))\
                       .orderBy('date')\
                       .show()

+----------+-----------------+
|      date|transaction_count|
+----------+-----------------+
|2024-01-15|                7|
|2024-01-16|                9|
|2024-01-17|               10|
|2024-01-18|               10|
|2024-01-19|                9|
|2024-01-20|               10|
|2024-01-21|               10|
|2024-01-22|                8|
|2024-01-23|               10|
|2024-01-24|               10|
|2024-01-25|                9|
|2024-01-26|               10|
|2024-01-27|               10|
|2024-01-28|               10|
|2024-01-29|               10|
|2024-01-30|               10|
|2024-01-31|               10|
|2024-02-01|               10|
|2024-02-02|               10|
|2024-02-03|               10|
+----------+-----------------+
only showing top 20 rows


### **6. Data Integration & Advanced Features**
* Join Operations: Join transactional data with a reference dataset (Product name, Brand) using optimization techniques like Broadcast Joins.
* Tracking: Utilize Accumulators to track specific metrics (e.g., total invalid records) across the cluster.

In [12]:
from pyspark.sql.functions import broadcast

df_joined = df_transactions_cleaned.join(
    broadcast(df_products_cleaned),
    on= 'product_id',
    how = 'inner'
)

df_joined.show()

+----------+--------------+--------+-------------+-------+--------+---------------------+--------+-------+--------------------+------------+
|product_id|transaction_id| user_id|     category|  price|quantity|transaction_timestamp|platform|country|        product_name|       brand|
+----------+--------------+--------+-------------+-------+--------+---------------------+--------+-------+--------------------+------------+
|   PROD028|        TXN128|USER1128|       Sports| 189.99|       1|  2024-01-27 15:10:45|     Web|Germany|  Basketball Premium|  SportsGear|
|   PROD030|        TXN130|USER1130|        Books|  29.99|       1|  2024-01-27 17:15:30|     Web|     UK|Cookbook Mediterr...|   ChefBooks|
|   PROD096|        TXN096|USER1096|  Electronics|  725.0|       1|  2024-01-24 13:15:20|     Web|    USA|       Router WiFi 6|    NetSpeed|
|   PROD064|        TXN164|USER1164|       Sports|  395.5|       1|  2024-01-31 11:45:30|     Web| Canada|Fishing Rod Profe...|   AnglerPro|
|   PROD048| 

### **7. Output Management**
* Write the final processed dataset to the local filesystem.
* The output must use a columnar storage format.
* Partitioning: Organize the output folder by a logical business column.

In [15]:
df_joined.coalesce(4) \
         .write\
         .mode('overwrite')\
         .option("compression", "snappy")\
         .partitionBy('country')\
         .parquet('processed_data')